# Database Query & Insert Tutorial

**Concise guide to querying and inserting data into the tracking analytics database.**

This notebook covers:
- Connecting to DuckDB and PostgreSQL
- Querying sessions, episodes, and observations
- Querying extended properties
- Inserting data (DuckDB only)
- Using the high-level QueryBackend API

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

from collab_env.data.db.config import DBConfig, get_db_config
from collab_env.data.db.db_loader import Boids3DLoader, DatabaseConnection
from collab_env.data.db.query_backend import QueryBackend

## Part 1: Connecting to Databases

The system supports both **DuckDB** (local file) and **PostgreSQL** (server).

In [2]:
# Connect to DuckDB (local file - for testing and insertions)
duckdb_path = "/tmp/test_tutorial.duckdb"

# Set environment variable before creating config
os.environ["DUCKDB_PATH"] = duckdb_path

duckdb_config = get_db_config(backend="duckdb")

db_duckdb = DatabaseConnection(duckdb_config)
db_duckdb.connect()
print(f"✓ Connected to DuckDB: {duckdb_path}")

2025-11-13 17:02:58 | INFO     | collab_env.data.db.db_loader:connect:113 - Connected to DuckDB: /tmp/test_tutorial.duckdb


✓ Connected to DuckDB: /tmp/test_tutorial.duckdb


In [3]:
# Connect to PostgreSQL (if available - for production queries)
# Skip this cell if you don't have PostgreSQL running
try:
    # Set environment variables for PostgreSQL
    os.environ["DB_BACKEND"] = "postgres"
    os.environ["POSTGRES_DB"] = "tracking_analytics"
    os.environ["POSTGRES_USER"] = "postgres"  # TODO: change to your username
    os.environ["POSTGRES_PASSWORD"] = "password"  # TODO: change to your password

    postgres_config = get_db_config(backend="postgres")

    db_postgres = DatabaseConnection(postgres_config)
    db_postgres.connect()
    print("✓ Connected to PostgreSQL: tracking_analytics")
except Exception as e:
    print(f"⚠ PostgreSQL not available: {e}")
    db_postgres = None

2025-11-13 17:03:02 | INFO     | collab_env.data.db.db_loader:connect:111 - Connected to PostgreSQL: tracking_analytics


✓ Connected to PostgreSQL: tracking_analytics


## Part 2: Initializing Test Database

Create tables and seed data in DuckDB for testing.

In [4]:
# Initialize DuckDB with schema
from collab_env.data.db.init_database import DatabaseBackend, get_schema_files
from collab_env.data.file_utils import get_project_root

# Get schema files
project_root = get_project_root()
schema_dir = project_root / "schema"
schema_files = get_schema_files(schema_dir)

# Create backend and execute schema
backend = DatabaseBackend(duckdb_config)
backend.connect()

for schema_file in schema_files:
    print(f"Executing {schema_file.name}...")
    backend.execute_file(schema_file)

backend.close()
print("✓ DuckDB schema initialized")

2025-11-13 17:03:08 | SUCCESS  | Connected to DuckDB: /tmp/test_tutorial.duckdb
2025-11-13 17:03:08 | SUCCESS  | Executed 01_core_tables.sql
2025-11-13 17:03:08 | SUCCESS  | Executed 02_extended_properties.sql
2025-11-13 17:03:08 | SUCCESS  | Executed 03_seed_data.sql


Executing 01_core_tables.sql...
Executing 02_extended_properties.sql...
Executing 03_seed_data.sql...
✓ DuckDB schema initialized


## Part 3: Inserting Data (DuckDB Only)

Insert sample session, episode, and observations data.

In [5]:
# Insert a test session
import json

session_data = {
    "session_id": "test-session-001",
    "session_name": "Tutorial Example Session",
    "category_id": "boids_3d",
    "config": json.dumps({"num_agents": 10, "scene_size": 480}),
    "metadata": json.dumps({"notes": "Created in tutorial notebook"}),
}

db_duckdb.execute(
    """
    INSERT INTO sessions (session_id, session_name, category_id, config, metadata)
    VALUES (:session_id, :session_name, :category_id, :config, :metadata)
    """,
    session_data,
)
print("✓ Inserted session")

✓ Inserted session


In [6]:
# Insert a test episode
episode_data = {
    "episode_id": "test-episode-001",
    "session_id": "test-session-001",
    "episode_number": 0,
    "num_frames": 100,
    "num_agents": 10,
    "frame_rate": 30.0,
    "file_path": "/tmp/test_episode.parquet",
}

db_duckdb.execute(
    """
    INSERT INTO episodes (episode_id, session_id, episode_number, num_frames, num_agents, frame_rate, file_path)
    VALUES (:episode_id, :session_id, :episode_number, :num_frames, :num_agents, :frame_rate, :file_path)
    """,
    episode_data,
)
print("✓ Inserted episode")

✓ Inserted episode


In [7]:
# Insert test observations using pandas (bulk insert)
num_agents = 10
num_frames = 100

# Generate synthetic trajectory data
observations = []
for time_idx in range(num_frames):
    for agent_id in range(num_agents):
        # Simple circular motion
        angle = 2 * np.pi * time_idx / num_frames + agent_id * 0.2
        radius = 100 + agent_id * 10

        x = 240 + radius * np.cos(angle)
        y = 240 + radius * np.sin(angle)
        z = 50 + 20 * np.sin(angle * 2)

        v_x = -radius * np.sin(angle) * 2 * np.pi / num_frames
        v_y = radius * np.cos(angle) * 2 * np.pi / num_frames
        v_z = 40 * np.cos(angle * 2) * 2 * np.pi / num_frames

        observations.append(
            {
                "episode_id": "test-episode-001",
                "time_index": time_idx,
                "agent_id": agent_id,
                "agent_type_id": "agent",
                "x": x,
                "y": y,
                "z": z,
                "v_x": v_x,
                "v_y": v_y,
                "v_z": v_z,
            }
        )

obs_df = pd.DataFrame(observations)
db_duckdb.insert_dataframe(obs_df, "observations", if_exists="append")
print(f"✓ Inserted {len(obs_df)} observations")

✓ Inserted 1000 observations


In [ ]:
# Insert extended properties (distance to target)
# First, get observation IDs
obs_ids = db_duckdb.fetch_all(
    """
    SELECT observation_id, time_index, agent_id
    FROM observations
    WHERE episode_id = :episode_id
    ORDER BY time_index, agent_id
    """,
    {"episode_id": "test-episode-001"},
)

# Compute synthetic distance to target center
target_center = np.array([240, 240, 50])
extended_props = []

for obs_id, time_idx, agent_id in obs_ids:
    # Get position from observations
    obs = obs_df[
        (obs_df["time_index"] == time_idx) & (obs_df["agent_id"] == agent_id)
    ].iloc[0]
    pos = np.array([obs["x"], obs["y"], obs["z"]])
    distance = np.linalg.norm(pos - target_center)

    extended_props.append(
        {
            "observation_id": obs_id,
            "property_id": "distance_to_target_center",
            "value_float": distance,
            "value_text": None,
        }
    )

ext_df = pd.DataFrame(extended_props)
db_duckdb.insert_dataframe(ext_df, "extended_properties", if_exists="append")
print(f"✓ Inserted {len(ext_df)} extended properties")

## Part 4: Basic Queries

Query the database using low-level SQL.

In [ ]:
# Query 1: List all sessions
sessions = db_duckdb.fetch_all("SELECT * FROM sessions")
print("Sessions:")
for session in sessions:
    print(f"  - {session[0]}: {session[1]} ({session[2]})")

In [ ]:
# Query 2: Get episodes for a session
episodes = db_duckdb.fetch_all(
    """
    SELECT episode_id, episode_number, num_frames, num_agents, frame_rate
    FROM episodes
    WHERE session_id = :session_id
    ORDER BY episode_number
    """,
    {"session_id": "test-session-001"},
)

print("\nEpisodes for test-session-001:")
for ep in episodes:
    print(f"  - {ep[0]}: {ep[2]} frames, {ep[3]} agents @ {ep[4]} fps")

In [ ]:
# Query 3: Get observations with computed speed
obs_query = """
SELECT 
    time_index,
    agent_id,
    x, y, z,
    v_x, v_y, v_z,
    sqrt(v_x*v_x + v_y*v_y + v_z*v_z) as speed
FROM observations
WHERE episode_id = :episode_id
  AND time_index < 5
ORDER BY time_index, agent_id
"""

from sqlalchemy import text

with db_duckdb.engine.connect() as conn:
    result = conn.execute(text(obs_query), {"episode_id": "test-episode-001"})
    obs_df_query = pd.DataFrame(result.fetchall(), columns=result.keys())

print("\nFirst 5 frames of observations:")
print(
    obs_df_query[["time_index", "agent_id", "x", "y", "z", "speed"]].to_string(
        index=False
    )
)

In [ ]:
# Query 4: Get observations with extended properties
extended_query = """
SELECT 
    o.time_index,
    o.agent_id,
    o.x, o.y, o.z,
    pd.property_name,
    ep.value_float
FROM observations o
JOIN extended_properties ep ON o.observation_id = ep.observation_id
JOIN property_definitions pd ON ep.property_id = pd.property_id
WHERE o.episode_id = :episode_id
  AND o.time_index < 5
ORDER BY o.time_index, o.agent_id
"""

with db_duckdb.engine.connect() as conn:
    result = conn.execute(text(extended_query), {"episode_id": "test-episode-001"})
    ext_query_df = pd.DataFrame(result.fetchall(), columns=result.keys())

print("\nObservations with extended properties (first 5 frames):")
print(ext_query_df.to_string(index=False))

In [ ]:
# Query 5: Aggregate statistics
stats_query = """
SELECT 
    COUNT(*) as total_observations,
    COUNT(DISTINCT agent_id) as num_agents,
    COUNT(DISTINCT time_index) as num_frames,
    AVG(sqrt(v_x*v_x + v_y*v_y + v_z*v_z)) as avg_speed,
    MAX(sqrt(v_x*v_x + v_y*v_y + v_z*v_z)) as max_speed
FROM observations
WHERE episode_id = :episode_id
"""

stats = db_duckdb.fetch_one(stats_query, {"episode_id": "test-episode-001"})
print("\nEpisode Statistics:")
print(f"  Total observations: {stats[0]}")
print(f"  Num agents: {stats[1]}")
print(f"  Num frames: {stats[2]}")
print(f"  Avg speed: {stats[3]:.2f}")
print(f"  Max speed: {stats[4]:.2f}")

## Part 5: QueryBackend API (Dashboard Pattern)

The **QueryBackend** provides high-level methods for common queries. This is how the dashboard uses it.

In [ ]:
# Initialize QueryBackend with DuckDB
query = QueryBackend(config=duckdb_config)
print("✓ QueryBackend initialized")

### 5.1 Session and Episode Discovery

In [ ]:
# Get all categories
categories = query.get_categories()
print("Categories:")
print(categories[["category_id", "category_name"]].to_string(index=False))

In [ ]:
# Get sessions by category
sessions = query.get_sessions(category_id="boids_3d")
print("\nBoids 3D Sessions:")
print(sessions[["session_id", "session_name", "category_id"]].to_string(index=False))

In [ ]:
# Get episodes for a session
episodes = query.get_episodes("test-session-001")
print("\nEpisodes:")
print(
    episodes[["episode_id", "num_frames", "num_agents", "frame_rate"]].to_string(
        index=False
    )
)

### 5.2 Spatial Analysis

In [ ]:
# Get spatial heatmap (binned positions)
heatmap = query.get_spatial_heatmap(
    episode_id="test-episode-001", bin_size=50.0, agent_type="agent"
)
print("\nSpatial Heatmap (top 10 bins by density):")
top_bins = heatmap.nlargest(10, "density")[["x_bin", "y_bin", "z_bin", "density"]]
print(top_bins.to_string(index=False))

In [ ]:
# Get episode tracks for visualization
tracks = query.get_episode_tracks(
    episode_id="test-episode-001", start_time=0, end_time=10
)
print("\nTracks (first 10 frames):")
print(
    tracks[["agent_id", "time_index", "x", "y", "z", "speed"]]
    .head(20)
    .to_string(index=False)
)

### 5.3 Episode Tracks (for visualization)

In [ ]:
# Get available extended properties
props = query.get_available_properties("test-episode-001")
print("\nAvailable Extended Properties:")
print(
    props[["property_id", "property_name", "data_type", "unit"]].to_string(index=False)
)

In [ ]:
# Get property distributions for histogram
dist = query.get_property_distributions(
    episode_id="test-episode-001", property_ids=["distance_to_target_center"]
)
print("\nDistance to Target Distribution:")
print(f"  Count: {len(dist)}")
print(f"  Mean: {dist['value_float'].mean():.2f}")
print(f"  Std: {dist['value_float'].std():.2f}")
print(f"  Min: {dist['value_float'].min():.2f}")
print(f"  Max: {dist['value_float'].max():.2f}")

### 5.4 Extended Properties

In [ ]:
# Get extended properties time series (windowed)
timeseries = query.get_extended_properties_timeseries(
    episode_id="test-episode-001",
    window_size=20,
    property_ids=["distance_to_target_center"],
)
print("\nExtended Properties Time Series (20-frame windows):")
print(
    timeseries[["time_window", "property_id", "avg_value", "std_value"]].to_string(
        index=False
    )
)

### 5.6 Dashboard Pattern: Using AnalysisContext

The dashboard uses `AnalysisContext` to share query parameters across widgets. This pattern enables:
- Consistent parameters across multiple analyses
- Easy parameter overrides for widget-specific customization
- Centralized scope management (episode/session level)

In [ ]:
# Import context classes (dashboard pattern)
from collab_env.dashboard.widgets import AnalysisContext, QueryScope, ScopeType

# Create a query scope for an episode
scope = QueryScope(
    scope_type=ScopeType.EPISODE,
    episode_id="test-episode-001",
    session_id="test-session-001",
    start_time=0,
    end_time=50,
    agent_type="agent",
)

# Create analysis context with shared parameters
context = AnalysisContext(
    query_backend=query,
    scope=scope,
    spatial_bin_size=20.0,  # Shared spatial discretization
    temporal_window_size=10,  # Shared time window
    min_samples=10,  # Shared minimum sample threshold
    on_loading=lambda msg: print(f"⏳ {msg}"),
    on_success=lambda msg: print(f"✓ {msg}"),
    on_error=lambda msg: print(f"✗ {msg}"),
)

print("✓ Created AnalysisContext")
print(f"  Scope: {scope.scope_type.value}")
print(f"  Episode: {scope.episode_id}")
print(f"  Time range: {scope.start_time}-{scope.end_time}")
print(f"  Spatial bin: {context.spatial_bin_size}")
print(f"  Time window: {context.temporal_window_size}")

In [ ]:
# Use context to get merged query parameters
params = context.get_query_params()
print("\nMerged Query Parameters:")
for key, value in params.items():
    print(f"  {key}: {value}")

# Query using merged parameters (dashboard pattern)
context.report_loading("Loading spatial heatmap...")
heatmap = query.get_spatial_heatmap(**params)
context.report_success(f"Loaded {len(heatmap)} bins")

print("\nHeatmap with context parameters:")
print(heatmap[["x_bin", "y_bin", "z_bin", "density"]].head().to_string(index=False))

In [ ]:
# Override specific parameters (widget-specific customization)
custom_params = context.get_query_params(
    bin_size=10.0, min_count=5  # Override spatial bin size  # Override minimum count
)

print("\nCustom Parameters (with overrides):")
print(f"  bin_size: {custom_params['bin_size']} (was {params['bin_size']})")
print(f"  min_count: {custom_params['min_count']} (was {params.get('min_count', 1)})")

# Query with custom parameters
custom_heatmap = query.get_spatial_heatmap(**custom_params)
print(f"\nCustom heatmap: {len(custom_heatmap)} bins (finer resolution)")

## Part 6: Querying PostgreSQL (if available)

Same queries work on PostgreSQL with production data.

In [ ]:
if db_postgres is not None:
    # Initialize QueryBackend for PostgreSQL
    postgres_config = get_db_config(backend="postgres")
    query_pg = QueryBackend(config=postgres_config)

    # Get sessions
    sessions_pg = query_pg.get_sessions(category_id="boids_3d")
    print("PostgreSQL - Boids 3D Sessions:")
    print(f"  Found {len(sessions_pg)} sessions")

    if len(sessions_pg) > 0:
        # Get first session's episodes
        session_id = sessions_pg.iloc[0]["session_id"]
        episodes_pg = query_pg.get_episodes(session_id)
        print(f"\n  Episodes for {session_id}:")
        print(f"    Found {len(episodes_pg)} episodes")

        if len(episodes_pg) > 0:
            # Get spatial heatmap
            episode_id = episodes_pg.iloc[0]["episode_id"]
            heatmap_pg = query_pg.get_spatial_heatmap(episode_id, bin_size=20.0)
            print(f"\n  Heatmap for {episode_id}:")
            print(f"    Generated {len(heatmap_pg)} bins")

    query_pg.close()
else:
    print("PostgreSQL not available - skipping")

## Part 7: Loading Real Data

Load actual simulation data using data loaders.

In [ ]:
# Example: Load 3D boids simulation (if data exists)
# Uncomment and modify path as needed

# sim_dir = Path("simulated_data/hackathon/hackathon-boid-small-200-sim_run-started-20250926-220926")
# if sim_dir.exists():
#     loader = Boids3DLoader(db_duckdb, max_episodes=2)  # Load only 2 episodes
#     loader.load_simulation(sim_dir)
#     print("✓ Loaded simulation data")
# else:
#     print("Simulation directory not found")

## Cleanup

In [ ]:
# Close connections
query.close()
db_duckdb.close()
if db_postgres is not None:
    db_postgres.close()

print("✓ All connections closed")

## Summary

### Key Takeaways

1. **Connecting**:
   - Set environment variables (`DUCKDB_PATH`, `DB_BACKEND`, etc.)
   - Use `get_db_config()` to create config from environment
   - Use `DatabaseConnection` for low-level access
   - Use `QueryBackend()` for high-level queries (no args needed!)

2. **Inserting** (DuckDB only):
   - `db.execute()` for single inserts with named parameters
   - `db.insert_dataframe()` for bulk inserts (much faster)
   - Use transactions for multi-operation consistency

3. **Querying with QueryBackend**:
   - **Discovery**: `get_categories()`, `get_sessions()`, `get_episodes()`
   - **Spatial**: `get_spatial_heatmap()`, `get_episode_tracks()`
   - **Properties**: `get_available_properties()`, `get_property_distributions()`, `get_extended_properties_timeseries()`
   - All methods return pandas DataFrames
   - All methods support optional time/agent filtering

4. **Advanced Pattern (Dashboard)**:
   - Create `QueryScope` to define what data to analyze
   - Create `AnalysisContext` with shared parameters
   - Use `context.get_query_params()` to merge scope + shared + custom params
   - Pass merged params to QueryBackend methods
   - Enables consistent parameters across multiple widgets/analyses

5. **Best Practices**:
   - Use DuckDB for testing and local development
   - Use PostgreSQL for production and Grafana
   - Use QueryBackend for cleaner, higher-level code
   - Use AnalysisContext for multi-widget applications
   - Always filter by episode_id for performance

### Next Steps

- [docs/data/db/README.md](README.md) - Complete database documentation
- [schema/README.md](../../../schema/README.md) - Database schema details
- [collab_env/dashboard/widgets/](../../dashboard/widgets/) - Widget examples using AnalysisContext
- [collab_env/data/db/queries/](../../data/db/queries/) - SQL query library